① データを受け取る

In [1]:
# ① Google Drive をマウント（最初に1回だけ）
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

# ① 参照フォルダを指定
base_path = "/content/drive/MyDrive/Kaggle研究会/playground-series-s6e1"

# ① CSVを読み込む
train = pd.read_csv(f"{base_path}/train.csv")
test = pd.read_csv(f"{base_path}/test.csv")
sample_sub = pd.read_csv(f"{base_path}/sample_submission.csv")

# 読み込み確認
print(train.shape, test.shape, sample_sub.shape)
train.head()

Mounted at /content/drive
(630000, 13) (270000, 12) (270000, 2)


,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,0,21,female,b.sc,7.91,98.8,no,4.9,average,online videos,low,easy,78.3
1,1,18,other,diploma,4.95,94.8,yes,4.7,poor,self-study,medium,moderate,46.7
2,2,20,female,b.sc,4.68,92.6,yes,5.8,poor,coaching,high,moderate,99.0
3,3,19,male,b.sc,2.00,49.5,yes,8.3,average,group study,high,moderate,63.9
4,4,23,male,bca,7.65,86.9,yes,9.6,good,self-study,high,easy,100.0


② 数値とカテゴリを分ける
*   予測したい列（目的変数）を y にする
*   モデルに入れる説明変数を X にする
*   id は識別子なので除外
*   特徴量を追加
*   前処理のために「数値列」「カテゴリ列」を分離

In [20]:
# ② 特徴量 X と目的変数 y
X = train.drop(["exam_score", "id"], axis=1)
y = train["exam_score"]

# 特徴量追加
# 数値×数値（study_sleep_ratio） → 効果小
# X["study_sleep_ratio"] = X["study_hours"] / (X["sleep_hours"] + 1)

# 数値×カテゴリ（difficulty_study） → 効果小
# 難易度を数値化して勉強時間と掛ける
# difficulty_map = {"easy": 1, "moderate": 2, "hard": 3}
# X["difficulty_study"] = X["exam_difficulty"].map(difficulty_map) * X["study_hours"]

# ② 数値列・カテゴリ列を分ける
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

print("数値特徴量:", list(numeric_features))
print("カテゴリ特徴量:", list(categorical_features))

数値特徴量: ['age', 'study_hours', 'class_attendance', 'sleep_hours', 'difficulty_study']
カテゴリ特徴量: ['gender', 'course', 'internet_access', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty']


③ カテゴリを数値に変換
* 機械学習モデルは文字（Male / Female など）を理解できない
* カテゴリ列を One-Hot Encoding で数値に変換する準備
* テストデータに未知カテゴリが出てもエラーにならない設定

In [21]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# ③ 数値はそのまま、カテゴリは One-Hot 化
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

④ 特徴量をまとめる
* ③の前処理と⑤のモデルを Pipeline で1本につなぐ
* 「学習時」と「予測時」で処理がズレる事故を防ぐ
* Kaggle・実務で必須の書き方

In [22]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor

# ④（⑤の準備）モデル定義
model = HistGradientBoostingRegressor(
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

# ④ 前処理 → モデル を1つにまとめる
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

追加① 今の特徴量（元の列）を確認する
* モデルに入れている“生の特徴量”を、数値/カテゴリに分けて確認

In [9]:
print("元の特徴量（X.columns）:", list(X.columns))
print("数値特徴量:", list(numeric_features))
print("カテゴリ特徴量:", list(categorical_features))

元の特徴量（X.columns）: ['age', 'gender', 'course', 'study_hours', 'class_attendance', 'internet_access', 'sleep_hours', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty']
数値特徴量: ['age', 'study_hours', 'class_attendance', 'sleep_hours']
カテゴリ特徴量: ['gender', 'course', 'internet_access', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty']


追加② One-Hot後の特徴量数と特徴量名（先頭だけ）を確認する
* カテゴリをOne-Hotにした結果、最終的にモデルが見る特徴量が何個になるかと、その名前の例を確認

In [10]:
# 前処理をfitして、One-Hot後の特徴量名を取得
preprocessor.fit(X)
feature_names = preprocessor.get_feature_names_out()

print("One-Hot後の特徴量数:", len(feature_names))
print("One-Hot後の特徴量名（先頭30）:", feature_names[:30])

One-Hot後の特徴量数: 30
One-Hot後の特徴量名（先頭30）: ['num__age' 'num__study_hours' 'num__class_attendance' 'num__sleep_hours'
 'cat__gender_female' 'cat__gender_male' 'cat__gender_other'
 'cat__course_b.com' 'cat__course_b.sc' 'cat__course_b.tech'
 'cat__course_ba' 'cat__course_bba' 'cat__course_bca'
 'cat__course_diploma' 'cat__internet_access_no'
 'cat__internet_access_yes' 'cat__sleep_quality_average'
 'cat__sleep_quality_good' 'cat__sleep_quality_poor'
 'cat__study_method_coaching' 'cat__study_method_group study'
 'cat__study_method_mixed' 'cat__study_method_online videos'
 'cat__study_method_self-study' 'cat__facility_rating_high'
 'cat__facility_rating_low' 'cat__facility_rating_medium'
 'cat__exam_difficulty_easy' 'cat__exam_difficulty_hard'
 'cat__exam_difficulty_moderate']


⑤ モデルで学習
* データを「学習用」と「検証用」に分ける
* 学習用データでモデルを訓練する
* 検証用データは 成績表（RMSE）用

In [23]:
from sklearn.model_selection import train_test_split

# ⑤ 学習用 / 検証用に分割
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ⑤ 学習
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  Index(['age', 'study_hours', 'class_attendance', 'sleep_hours',
       'difficulty_study'],
      dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['gender', 'course', 'internet_access', 'sleep_quality', 'study_method',
       'facility_rating', 'exam_difficulty'],
      dtype='object'))])),
                ('model',
                 HistGradientBoostingRegressor(max_depth=6, random_state=42))])

⑥ 予測
* 学習に使っていない検証データで予測（精度確認用）
* Kaggle提出用の test データでも予測を作る

In [24]:
# ⑥ 検証データで予測
valid_pred = pipeline.predict(X_valid)

# ⑥ テストデータで予測（id は除外）
test_X = test.drop("id", axis=1)
# 数値×数値（study_sleep_ratio） → 効果小
# test_X["study_sleep_ratio"] = test_X["study_hours"] / (test_X["sleep_hours"] + 1)

# 数値×カテゴリdifficulty_study（exam_difficulty × study_hours）→ 効果小
# difficulty_map = {"easy": 1, "moderate": 2, "hard": 3}
# test_X["difficulty_study"] = (
#     test_X["exam_difficulty"].map(difficulty_map) * test_X["study_hours"]
# )
test_pred = pipeline.predict(test_X)

⑦ RMSEで評価
* Kaggleと同じ評価指標 RMSE を計算
* 「平均で何点ズレているか」を数値で確認

In [25]:
from sklearn.metrics import mean_squared_error
import numpy as np

# ⑦ RMSEを計算
rmse = np.sqrt(mean_squared_error(y_valid, valid_pred))
rmse

np.float64(8.80123593409469)

⑧ 提出用CSVを作成
* Kaggle指定フォーマットに予測結果を入れる
* 提出可能な submission.csv を作る

In [18]:
# （参考）提出用CSVを作成し、指定フォルダに保存する

# 保存先フォルダ
submit_path = "/content/drive/MyDrive/Kaggle研究会/2026年1月提出物"

# submission 作成
submission = sample_sub.copy()
submission["exam_score"] = test_pred

# CSVとして保存
submission.to_csv(f"{submit_path}/submission20250118_02.csv", index=False)

# 確認
submission.head()

,id,exam_score
0,630000,71.494564
1,630001,69.699981
2,630002,87.613141
3,630003,54.195662
4,630004,48.106754
